# 🛡️ Infosys Springboard Internship 7.0 (Batch 4) - Milestone 1
## User Authentication Module &bull; Streamlit + JWT + Gmail OTP + SQLite + ngrok

This notebook builds and launches a production-ready, security-hardened authentication portal with:
1. **Login**: Dual login via Username or Email with generic error masking.
2. **Signup**: 6 mandatory fields, unique username rejection, and redirection back to Login.
3. **Field Validation**: Milestone-compliant email regex (`ab@cd.ef`) and password complexity rules.
4. **JWT Session Management**: Signed session tokens with 2-hour expiration.
5. **Forgot Password**: Dual recovery routes (Security Question + 6-digit Email OTP with 5-min expiry).
6. **User Dashboard**: Welcome message with logged-in user profile, metric cards, and logout.
7. **Admin Dashboard**: Dedicated admin login listing registered users (passwords never exposed).
8. **Security**: Zero hardcoded secrets, salted bcrypt hashing, Colab Secrets integration.

---

### 🔑 Required Google Colab Secrets Setup (Before Running)
Click the **Key icon (Secrets)** in the left sidebar of Google Colab and add these 4 secrets with **Notebook Access toggled ON**:
* `JWT_SECRET` : Any long random secret string (used to sign JWT session and OTP tokens).
* `NGROK_AUTHTOKEN` : Your ngrok authentication token from https://dashboard.ngrok.com.
* `EMAIL_ADDRESS` : Your Gmail address that will send OTP verification emails.
* `EMAIL_PASSWORD` : A 16-character Google **App Password** (Google Account -> Security -> 2-Step Verification -> App Passwords).


In [ ]:
# 1. Install Dependencies
!pip install -q streamlit pyjwt bcrypt pyngrok streamlit-option-menu pytest pandas


In [ ]:
# 2. Verify Google Colab Secrets
from google.colab import userdata

REQUIRED_SECRETS = ['JWT_SECRET', 'NGROK_AUTHTOKEN', 'EMAIL_ADDRESS', 'EMAIL_PASSWORD']
missing_secrets = []

for secret_name in REQUIRED_SECRETS:
    try:
        val = userdata.get(secret_name)
        if not val or str(val).strip() == '':
            missing_secrets.append(secret_name)
    except Exception:
        missing_secrets.append(secret_name)

if missing_secrets:
    print('❌ The following required secrets are missing in your Colab Secrets tab:')
    for s in missing_secrets:
        print(f'   - {s}')
    print('\nPlease click the Key icon (Secrets) in the left sidebar, add each secret, and toggle Notebook access ON.')
else:
    print('✅ All required Colab Secrets are configured and accessible!')


In [ ]:
import os
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('''[theme]
base = "light"
primaryColor = "#0d9488"
backgroundColor = "#faf9f6"
secondaryBackgroundColor = "#f1f5f9"
textColor = "#0f172a"
font = "sans serif"

[server]
headless = true
enableCORS = false
enableXsrfProtection = false
maxUploadSize = 10
''')
print("✅ .streamlit/config.toml configured.")


In [ ]:
%%writefile /content/Milestone1/config.py
"""
Configuration and secrets management module for Milestone 1.

Reads sensitive keys at runtime:
- In Google Colab: strictly reads from Google Colab Secrets (`google.colab.userdata`).
  If a secret is missing, it provides clear setup instructions without leaking data.
- In local development/testing: reads from environment variables or uses
  explicit test-only dummy placeholders.

Secret names used (configured manually by user in Colab Secrets tab):
- JWT_SECRET
- NGROK_AUTHTOKEN
- EMAIL_ADDRESS
- EMAIL_PASSWORD
"""

import os
import sys
from typing import Dict, List, Optional, Tuple


def is_running_in_colab() -> bool:
    """Check if code is running inside a Google Colab runtime environment."""
    return "google.colab" in sys.modules or os.path.exists("/content")


def get_secret(name: str) -> Optional[str]:
    """
    Retrieve secret securely at runtime.
    In Colab: reads strictly from google.colab.userdata.
    In Local: reads from environment variables.
    Never returns hardcoded real credentials.
    """
    # 1. Check Google Colab Secrets
    try:
        from google.colab import userdata  # type: ignore
        val = userdata.get(name)
        if val is not None and str(val).strip() != "":
            return str(val).strip()
    except Exception:
        pass

    # 2. Check environment variable (for local testing)
    env_val = os.getenv(name)
    if env_val is not None and env_val.strip() != "":
        return env_val.strip()

    return None


def get_jwt_secret() -> str:
    """
    Retrieve JWT signing secret.
    In Colab, requires JWT_SECRET secret.
    In local test runner, uses an explicit non-production test placeholder if not set.
    """
    secret = get_secret("JWT_SECRET")
    if secret:
        return secret
    if is_running_in_colab():
        raise RuntimeError("Missing required Colab Secret: 'JWT_SECRET'. Please add it in the Secrets tab.")
    # Explicit dummy placeholder strictly for local pytest execution
    return "TESTING_ONLY_LOCAL_JWT_KEY_NOT_FOR_PRODUCTION"


def get_ngrok_authtoken() -> Optional[str]:
    """Retrieve ngrok authtoken from Colab Secrets or environment."""
    return get_secret("NGROK_AUTHTOKEN")


def get_email_address() -> Optional[str]:
    """Retrieve Gmail address for OTP sender from Colab Secrets or environment."""
    return get_secret("EMAIL_ADDRESS")


def get_email_password() -> Optional[str]:
    """Retrieve Gmail App Password for OTP sender from Colab Secrets or environment."""
    return get_secret("EMAIL_PASSWORD")


def get_admin_credentials() -> Tuple[str, str]:
    """Load admin credentials securely from Colab Secrets or environment."""
    username = get_secret("ADMIN_USERNAME")
    password = get_secret("ADMIN_PASSWORD")

    if not username or not password:
        raise RuntimeError(
            "Missing ADMIN_USERNAME or ADMIN_PASSWORD. "
            "Configure both in Colab Secrets."
        )

    return username, password


def check_required_secrets_status() -> Tuple[bool, List[str]]:
    """
    Audit whether required secrets are present in the current runtime environment.
    Returns (all_present, missing_secret_names).
    """
    required = ["JWT_SECRET", "NGROK_AUTHTOKEN", "EMAIL_ADDRESS", "EMAIL_PASSWORD"]
    missing = [name for name in required if not get_secret(name)]
    return len(missing) == 0, missing


# OTP application settings
OTP_EXPIRY_MINUTES = 5
DEFAULT_DB_FILENAME = "users.db"



In [ ]:
%%writefile /content/Milestone1/database.py
"""
Unified SQLite database module for Milestone 1.

Provides a unified, thread-safe schema for signup, login,
security-question recovery, OTP verification tracking, and admin user listing.
"""

import os
import sqlite3
import time
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple

try:
    from Milestone1.config import DEFAULT_DB_FILENAME
except ImportError:
    from config import DEFAULT_DB_FILENAME

# Predefined fixed list of security questions as required by Step 3
SECURITY_QUESTIONS: List[str] = [
    "What was the name of your first elementary school?",
    "What is the name of your favorite childhood pet?",
    "In which city or town was your first job?",
    "What is your mother's maiden name?",
    "What was your childhood nickname?"
]


def resolve_db_path(db_path: Optional[str] = None) -> str:
    """Resolve the SQLite database path, defaulting to users.db in the Milestone1 folder."""
    if db_path:
        return db_path
    base_dir = os.path.dirname(os.path.abspath(__file__))
    return os.path.join(base_dir, DEFAULT_DB_FILENAME)


def get_connection(db_path: Optional[str] = None) -> sqlite3.Connection:
    """Create a thread-safe connection with row_factory configured."""
    path = resolve_db_path(db_path)
    conn = sqlite3.connect(path, check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn


def init_db(db_path: Optional[str] = None) -> None:
    """Initialize the unified database tables and indexes."""
    conn = get_connection(db_path)
    try:
        with conn:
            # 1. Main Users Table
            conn.execute("""
                CREATE TABLE IF NOT EXISTS users (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    username TEXT UNIQUE NOT NULL COLLATE NOCASE,
                    email TEXT UNIQUE NOT NULL COLLATE NOCASE,
                    password_hash TEXT NOT NULL,
                    security_question TEXT NOT NULL,
                    security_answer_hash TEXT NOT NULL,
                    created_at TEXT NOT NULL
                );
            """)

            # 2. OTP Tracking Table to enforce 5-minute expiry, single-use, and attempt limits
            conn.execute("""
                CREATE TABLE IF NOT EXISTS otp_tokens (
                    token_id TEXT PRIMARY KEY,
                    email TEXT NOT NULL COLLATE NOCASE,
                    otp_hash TEXT NOT NULL,
                    created_at REAL NOT NULL,
                    expires_at REAL NOT NULL,
                    is_used INTEGER DEFAULT 0,
                    attempts INTEGER DEFAULT 0
                );
            """)

            # Indexes for fast lookup
            conn.execute("CREATE INDEX IF NOT EXISTS idx_users_username ON users(username);")
            conn.execute("CREATE INDEX IF NOT EXISTS idx_users_email ON users(email);")
            conn.execute("CREATE INDEX IF NOT EXISTS idx_otp_email ON otp_tokens(email);")
    finally:
        conn.close()


def check_username_exists(username: str, db_path: Optional[str] = None) -> bool:
    """Check if a username is already taken."""
    conn = get_connection(db_path)
    try:
        row = conn.execute("SELECT 1 FROM users WHERE username = ?", (username.strip(),)).fetchone()
        return row is not None
    finally:
        conn.close()


def check_email_exists(email: str, db_path: Optional[str] = None) -> bool:
    """Check if an email is already registered."""
    conn = get_connection(db_path)
    try:
        row = conn.execute("SELECT 1 FROM users WHERE email = ?", (email.strip().lower(),)).fetchone()
        return row is not None
    finally:
        conn.close()


def create_user(
    username: str,
    email: str,
    password_hash: str,
    security_question: str,
    security_answer_hash: str,
    db_path: Optional[str] = None
) -> Tuple[bool, str]:
    """
    Register a new user in the unified database.
    Explicitly checks for duplicate username or email to give actionable feedback.
    """
    clean_username = username.strip()
    clean_email = email.strip().lower()

    if check_username_exists(clean_username, db_path=db_path):
        return False, f"Username '{clean_username}' is already taken. Please choose another username."

    if check_email_exists(clean_email, db_path=db_path):
        return False, f"Email '{clean_email}' is already registered. Please sign in or reset your password."

    created_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

    conn = get_connection(db_path)
    try:
        with conn:
            conn.execute(
                """
                INSERT INTO users (username, email, password_hash, security_question, security_answer_hash, created_at)
                VALUES (?, ?, ?, ?, ?, ?)
                """,
                (clean_username, clean_email, password_hash, security_question, security_answer_hash, created_at)
            )
        return True, "User registered successfully."
    except sqlite3.IntegrityError as e:
        err_msg = str(e).lower()
        if "username" in err_msg:
            return False, f"Username '{clean_username}' is already taken."
        elif "email" in err_msg:
            return False, f"Email '{clean_email}' is already registered."
        return False, "Registration failed due to a database integrity error."
    finally:
        conn.close()


def get_user_by_identifier(identifier: str, db_path: Optional[str] = None) -> Optional[Dict]:
    """
    Retrieve user record by either username OR email address.
    Enables unified login using username or email.
    """
    clean_id = identifier.strip()
    conn = get_connection(db_path)
    try:
        row = conn.execute(
            """
            SELECT id, username, email, password_hash, security_question, security_answer_hash, created_at
            FROM users
            WHERE username = ? OR email = ?
            """,
            (clean_id, clean_id.lower())
        ).fetchone()
        if row:
            return dict(row)
        return None
    finally:
        conn.close()


def get_user_by_username(username: str, db_path: Optional[str] = None) -> Optional[Dict]:
    """Retrieve user record specifically by username."""
    conn = get_connection(db_path)
    try:
        row = conn.execute(
            """
            SELECT id, username, email, password_hash, security_question, security_answer_hash, created_at
            FROM users
            WHERE username = ?
            """,
            (username.strip(),)
        ).fetchone()
        return dict(row) if row else None
    finally:
        conn.close()


def get_user_by_email(email: str, db_path: Optional[str] = None) -> Optional[Dict]:
    """Retrieve user record specifically by email."""
    conn = get_connection(db_path)
    try:
        row = conn.execute(
            """
            SELECT id, username, email, password_hash, security_question, security_answer_hash, created_at
            FROM users
            WHERE email = ?
            """,
            (email.strip().lower(),)
        ).fetchone()
        return dict(row) if row else None
    finally:
        conn.close()


def update_password(identifier: str, new_password_hash: str, db_path: Optional[str] = None) -> bool:
    """Update password hash for a user by username or email."""
    clean_id = identifier.strip()
    conn = get_connection(db_path)
    try:
        with conn:
            cursor = conn.execute(
                """
                UPDATE users
                SET password_hash = ?
                WHERE username = ? OR email = ?
                """,
                (new_password_hash, clean_id, clean_id.lower())
            )
            return cursor.rowcount > 0
    finally:
        conn.close()


def get_all_registered_users(db_path: Optional[str] = None) -> List[Dict]:
    """
    Fetch list of all registered users for the Admin Dashboard.
    CRITICAL: Never selects or exposes password_hash or security_answer_hash.
    """
    conn = get_connection(db_path)
    try:
        rows = conn.execute(
            """
            SELECT id, username, email, created_at
            FROM users
            ORDER BY id ASC
            """
        ).fetchall()
        return [dict(r) for r in rows]
    finally:
        conn.close()


# ── OTP Tracking and Single-Use Enforcement ────────────────────────────

def store_otp_record(
    token_id: str,
    email: str,
    otp_hash: str,
    expires_at: float,
    db_path: Optional[str] = None
) -> None:
    """Store an active OTP record with expiration and usage tracking."""
    conn = get_connection(db_path)
    try:
        with conn:
            conn.execute(
                """
                INSERT OR REPLACE INTO otp_tokens (token_id, email, otp_hash, created_at, expires_at, is_used, attempts)
                VALUES (?, ?, ?, ?, ?, 0, 0)
                """,
                (token_id, email.strip().lower(), otp_hash, time.time(), expires_at)
            )
    finally:
        conn.close()


def verify_and_consume_otp_record(
    token_id: str,
    input_otp: str,
    check_otp_fn,
    max_attempts: int = 5,
    db_path: Optional[str] = None
) -> Tuple[bool, str]:
    """
    Verify OTP with single-use consumption and brute-force attempt limits.
    Prevents replay attacks, repeated guesses, and expired OTP usage.
    """
    now = time.time()
    conn = get_connection(db_path)
    try:
        with conn:
            row = conn.execute(
                "SELECT email, otp_hash, expires_at, is_used, attempts FROM otp_tokens WHERE token_id = ?",
                (token_id,)
            ).fetchone()

            if not row:
                return False, "Invalid or non-existent verification session. Please request a new OTP."

            if row["is_used"] == 1:
                return False, "This OTP has already been used. Please request a fresh code."

            if now > row["expires_at"]:
                return False, "This verification code has expired (valid for 5 minutes). Please request a new one."

            if row["attempts"] >= max_attempts:
                return False, "Maximum verification attempts exceeded for this code. Please request a new OTP."

            # Verify the OTP hash using the provided checker function
            if check_otp_fn(input_otp, row["otp_hash"]):
                # Mark as consumed immediately
                conn.execute(
                    "UPDATE otp_tokens SET is_used = 1 WHERE token_id = ?",
                    (token_id,)
                )
                return True, "OTP verified successfully."
            else:
                # Increment failed attempts
                new_attempts = row["attempts"] + 1
                conn.execute(
                    "UPDATE otp_tokens SET attempts = ? WHERE token_id = ?",
                    (new_attempts, token_id)
                )
                remaining = max_attempts - new_attempts
                if remaining > 0:
                    return False, f"Incorrect verification code. {remaining} attempt(s) remaining."
                return False, "Maximum attempts exceeded. This OTP has been invalidated."
    finally:
        conn.close()


In [ ]:
%%writefile /content/Milestone1/auth.py
"""
Authentication, validation, hashing, and JWT session handling module for Milestone 1.

Strictly enforces Milestone 1 requirements:
- Mandatory field validations
- Email format rule (>=2 chars before @, >=2 chars between @ and dot, >=2 chars after dot)
- Password complexity rules (>=8 chars, uppercase, lowercase, digit, special symbol)
- bcrypt salted hashing for passwords and normalized security answers
- JWT session token generation, verification, and expiration handling
"""

import re
from datetime import datetime, timedelta, timezone
from typing import Any, Dict, Optional, Tuple

import bcrypt
import jwt

try:
    from Milestone1.config import get_jwt_secret
except ImportError:
    from config import get_jwt_secret

# Special symbols regex for password rule
SPECIAL_SYMBOLS_PATTERN = r"[!@#$%^&*(),.?\":{}|<>_\-+=\[\]\\/`~;']"


# ── Field & Format Validations ──────────────────────────────────────────

def validate_mandatory_fields(**fields: Any) -> Tuple[bool, str]:
    """
    Ensure no required field is empty or whitespace-only.
    Returns (True, "") or (False, "Field '<name>' cannot be empty.")
    """
    for name, value in fields.items():
        if value is None or (isinstance(value, str) and value.strip() == ""):
            formatted_name = name.replace("_", " ").capitalize()
            return False, f"The field '{formatted_name}' is mandatory and cannot be empty."
    return True, ""


def validate_email(email: str) -> Tuple[bool, str]:
    """
    Validate email address according to Milestone 1 Step 8:
    'At least 2 letters before the @, at least 2 letters between the @ and the dot,
    and at least 2 letters after the final dot — e.g. an address shaped like ab@cd.ef.'
    """
    if not email or not isinstance(email, str):
        return False, "Email address is required."

    clean_email = email.strip()

    # Must contain exactly one '@'
    if clean_email.count("@") != 1:
        return False, "Email must contain exactly one '@' symbol."

    local_part, domain_part = clean_email.split("@")

    # At least 2 characters before the @
    if len(local_part) < 2:
        return False, "Email must contain at least 2 characters before the '@' (e.g. 'ab@cd.ef')."

    # Domain part must have at least one dot
    if "." not in domain_part:
        return False, "Email domain must contain a dot (e.g. 'ab@cd.ef')."

    domain_segments = domain_part.rsplit(".", 1)
    before_last_dot = domain_segments[0]
    after_last_dot = domain_segments[1]

    # At least 2 characters between the @ and the final dot
    # If multiple dots exist, the last domain segment before the final dot (or entire prefix)
    # must have at least 2 chars.
    last_subdomain = before_last_dot.split(".")[-1]
    if len(last_subdomain) < 2 or len(before_last_dot) < 2:
        return False, "Email must contain at least 2 characters between the '@' and the dot (e.g. 'ab@cd.ef')."

    # At least 2 characters after the final dot
    if len(after_last_dot) < 2:
        return False, "Email must contain at least 2 characters after the final dot (e.g. 'ab@cd.ef')."

    # Standard RFC-like character safety regex check
    pattern = r"^[a-zA-Z0-9._%+-]{2,}@[a-zA-Z0-9.-]{2,}\.[a-zA-Z]{2,}$"
    if not re.match(pattern, clean_email):
        return False, "Email contains invalid characters or does not meet the format requirements."

    return True, ""


def validate_password(password: str) -> Tuple[bool, str]:
    """
    Validate password complexity according to Milestone 1 Step 8:
    - Minimum 8 characters
    - At least one uppercase letter
    - At least one lowercase letter
    - At least one number
    - At least one special symbol
    """
    if not password or not isinstance(password, str):
        return False, "Password is required."

    if len(password) < 8:
        return False, "Password must be at least 8 characters long."

    if not re.search(r"[A-Z]", password):
        return False, "Password must contain at least one uppercase letter."

    if not re.search(r"[a-z]", password):
        return False, "Password must contain at least one lowercase letter."

    if not re.search(r"[0-9]", password):
        return False, "Password must contain at least one number."

    if not re.search(SPECIAL_SYMBOLS_PATTERN, password):
        return False, "Password must contain at least one special symbol (e.g. @, #, $, %, !)."

    return True, ""


def validate_password_confirmation(password: str, confirm_password: str) -> Tuple[bool, str]:
    """Ensure password and confirm_password match exactly."""
    if password != confirm_password:
        return False, "Passwords do not match. Please re-enter your confirm password."
    return True, ""


# ── Password & Security Answer Hashing ──────────────────────────────────

def hash_password(password: str) -> str:
    """Hash password using bcrypt with a unique salt."""
    salt = bcrypt.gensalt(rounds=12)
    return bcrypt.hashpw(password.encode("utf-8"), salt).decode("utf-8")


def verify_password(plain_password: str, hashed_password: str) -> bool:
    """Verify plain password against a bcrypt hash."""
    if not plain_password or not hashed_password:
        return False
    try:
        return bcrypt.checkpw(
            plain_password.encode("utf-8"),
            hashed_password.encode("utf-8")
        )
    except Exception:
        return False


def normalize_security_answer(answer: str) -> str:
    """Normalize security answer by stripping whitespace and converting to lowercase."""
    if not answer:
        return ""
    # Collapse multiple internal spaces and trim
    return " ".join(answer.strip().lower().split())


def hash_security_answer(answer: str) -> str:
    """Hash normalized security answer using bcrypt."""
    normalized = normalize_security_answer(answer)
    salt = bcrypt.gensalt(rounds=12)
    return bcrypt.hashpw(normalized.encode("utf-8"), salt).decode("utf-8")


def verify_security_answer(plain_answer: str, hashed_answer: str) -> bool:
    """Verify plain security answer against stored bcrypt hash after normalization."""
    if not plain_answer or not hashed_answer:
        return False
    normalized = normalize_security_answer(plain_answer)
    try:
        return bcrypt.checkpw(
            normalized.encode("utf-8"),
            hashed_answer.encode("utf-8")
        )
    except Exception:
        return False


# ── JWT Session Management ──────────────────────────────────────────────

def create_session_jwt(
    username: str,
    email: str,
    role: str = "user",
    expires_in_hours: int = 2
) -> str:
    """
    Issue a signed JWT session token upon successful login (Step 7).
    """
    secret = get_jwt_secret()
    now = datetime.now(timezone.utc)
    payload = {
        "sub": username,
        "username": username,
        "email": email,
        "role": role,
        "iat": int(now.timestamp()),
        "exp": int((now + timedelta(hours=expires_in_hours)).timestamp())
    }
    return jwt.encode(payload, secret, algorithm="HS256")


def verify_session_jwt(token: str) -> Tuple[bool, Optional[Dict[str, Any]], str]:
    """
    Verify and decode a session JWT token.
    Returns (is_valid, payload, error_message).
    """
    if not token or not isinstance(token, str):
        return False, None, "Session token is missing."

    secret = get_jwt_secret()
    try:
        payload = jwt.decode(token, secret, algorithms=["HS256"])
        if "username" not in payload or "email" not in payload:
            return False, None, "Session token payload is invalid."
        return True, payload, "Token is valid."
    except jwt.ExpiredSignatureError:
        return False, None, "Session expired. Please log in again."
    except jwt.InvalidTokenError:
        return False, None, "Invalid session token. Please log in again."
    except Exception as e:
        return False, None, f"Token verification error: {str(e)}"


In [ ]:
%%writefile /content/Milestone1/otp_service.py
"""
OTP generation, JWT verification, and email dispatch service for Milestone 1.

Features:
- Cryptographically secure 6-digit OTP generation using secrets module
- Signed 5-minute JWT verification token
- Single-use consumption and attempt limiting via unified SQLite database
- Gmail SMTP delivery using TLS and anti-spam RFC headers
- Safe secret lookup from Colab Secrets without exposing credentials
"""

import datetime
import secrets
import smtplib
import uuid
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.utils import formatdate, make_msgid
from typing import Optional, Tuple

import jwt

try:
    from Milestone1.auth import hash_password, verify_password
    from Milestone1.config import (
        OTP_EXPIRY_MINUTES,
        get_email_address,
        get_email_password,
        get_jwt_secret,
    )
    from Milestone1.database import store_otp_record, verify_and_consume_otp_record
except ImportError:
    from auth import hash_password, verify_password
    from config import (
        OTP_EXPIRY_MINUTES,
        get_email_address,
        get_email_password,
        get_jwt_secret,
    )
    from database import store_otp_record, verify_and_consume_otp_record



def generate_otp() -> str:
    """Generate an unbiased, cryptographically secure 6-digit numerical OTP."""
    return f"{secrets.randbelow(900000) + 100000}"


def create_otp_session(email: str, otp: str, db_path: Optional[str] = None) -> Tuple[str, str]:
    """
    Create a secure OTP session:
    1. Generates a unique session token_id.
    2. Hashes the OTP using bcrypt.
    3. Persists OTP metadata in the database for single-use consumption and attempt limits.
    4. Issues a signed JWT containing token_id and otp_hash with a 5-minute expiration.
    Returns (jwt_token, token_id).
    """
    clean_email = email.strip().lower()
    token_id = str(uuid.uuid4())
    otp_hash = hash_password(otp)
    secret = get_jwt_secret()

    now = datetime.datetime.now(datetime.timezone.utc)
    exp = now + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)
    expires_at_timestamp = exp.timestamp()

    # Persist in SQLite for single-use and attempt enforcement
    store_otp_record(
        token_id=token_id,
        email=clean_email,
        otp_hash=otp_hash,
        expires_at=expires_at_timestamp,
        db_path=db_path
    )

    payload = {
        "sub": clean_email,
        "token_id": token_id,
        "type": "password_reset_otp",
        "iat": int(now.timestamp()),
        "exp": int(expires_at_timestamp)
    }

    jwt_token = jwt.encode(payload, secret, algorithm="HS256")
    return jwt_token, token_id


def verify_otp_session(
    jwt_token: str,
    input_otp: str,
    target_email: str,
    db_path: Optional[str] = None
) -> Tuple[bool, str]:
    """
    Verify OTP token:
    1. Validates JWT signature, expiration, and payload claims.
    2. Enforces single-use consumption and attempt counting in the database.
    Returns (is_valid, status_message).
    """
    if not jwt_token or not input_otp:
        return False, "Verification token and 6-digit OTP code are both required."

    secret = get_jwt_secret()
    clean_email = target_email.strip().lower()

    # Step 1: Decode JWT
    try:
        payload = jwt.decode(jwt_token, secret, algorithms=["HS256"])
    except jwt.ExpiredSignatureError:
        return False, f"This verification code expired after {OTP_EXPIRY_MINUTES} minutes. Please request a new code."
    except jwt.InvalidTokenError:
        return False, "Verification session token is invalid or corrupted. Please request a new code."
    except Exception as e:
        return False, f"Token verification error: {str(e)}"

    # Step 2: Validate payload claims
    if payload.get("sub") != clean_email or payload.get("type") != "password_reset_otp":
        return False, "Security token mismatch. Please request a new code."

    token_id = payload.get("token_id")
    if not token_id:
        return False, "Invalid verification token structure."

    # Step 3: Verify and consume against database (single-use & attempt-limited)
    return verify_and_consume_otp_record(
        token_id=token_id,
        input_otp=input_otp.strip(),
        check_otp_fn=verify_password,
        max_attempts=5,
        db_path=db_path
    )


def send_otp_email(to_email: str, otp: str) -> Tuple[bool, str]:
    """
    Send OTP verification email via Gmail SMTP using credentials from Colab Secrets.
    Returns (success, message).
    """
    sender_email = get_email_address()
    app_password = get_email_password()

    if not sender_email or not app_password:
        return False, (
            "Gmail credentials are not configured. "
            "Please add 'EMAIL_ADDRESS' and 'EMAIL_PASSWORD' to Google Colab Secrets (or environment) to send OTPs."
        )

    clean_to = to_email.strip().lower()

    # Build MIME message with anti-spam RFC headers
    msg = MIMEMultipart("alternative")
    msg["From"] = f"Infosys Portal <{sender_email}>"
    msg["To"] = clean_to
    msg["Subject"] = "Infosys Portal - Password Reset Verification Code"
    msg["Date"] = formatdate(localtime=True)
    msg["Message-ID"] = make_msgid()
    msg["Reply-To"] = sender_email

    # Plaintext fallback
    plain_text = (
        f"Infosys Portal - Password Reset Verification Code\n\n"
        f"Your one-time verification code is: {otp}\n\n"
        f"This code will expire in {OTP_EXPIRY_MINUTES} minutes.\n"
        f"If you did not request this password reset, please ignore this email.\n\n"
        f"Infosys Springboard Internship 7.0 - User Authentication Module"
    )

    # Professional HTML template matching the Light Theme (warm white, deep navy, teal)
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <style>
            body {{
                font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
                background-color: #faf9f6;
                margin: 0;
                padding: 24px;
                color: #0f172a;
            }}
            .card {{
                max-width: 520px;
                margin: 0 auto;
                background-color: #ffffff;
                border: 1px solid #e2e8f0;
                border-radius: 12px;
                padding: 32px;
                box-shadow: 0 4px 12px rgba(15, 23, 42, 0.05);
            }}
            .header {{
                text-align: center;
                border-bottom: 1px solid #f1f5f9;
                padding-bottom: 20px;
                margin-bottom: 24px;
            }}
            .title {{
                color: #0f172a;
                font-size: 22px;
                font-weight: 700;
                margin: 0;
            }}
            .subtitle {{
                color: #0d9488;
                font-size: 13px;
                font-weight: 600;
                text-transform: uppercase;
                letter-spacing: 1px;
                margin-top: 4px;
            }}
            .content {{
                font-size: 15px;
                line-height: 1.6;
                color: #334155;
            }}
            .otp-container {{
                text-align: center;
                margin: 28px 0;
            }}
            .otp-badge {{
                display: inline-block;
                background: #f0fdfa;
                color: #0d9488;
                border: 2px solid #0d9488;
                border-radius: 10px;
                font-size: 32px;
                font-weight: 800;
                letter-spacing: 8px;
                padding: 14px 28px;
            }}
            .expiry-note {{
                font-size: 13px;
                color: #64748b;
                text-align: center;
                margin-top: -12px;
                margin-bottom: 24px;
            }}
            .footer {{
                font-size: 12px;
                color: #94a3b8;
                border-top: 1px solid #f1f5f9;
                padding-top: 18px;
                text-align: center;
                line-height: 1.5;
            }}
        </style>
    </head>
    <body>
        <div class="card">
            <div class="header">
                <div class="title">Infosys Portal</div>
                <div class="subtitle">Secure Password Recovery</div>
            </div>
            <div class="content">
                <p>Hello,</p>
                <p>We received a password reset request for your registered account (<b>{clean_to}</b>). Please enter the 6-digit verification code below to proceed:</p>
                <div class="otp-container">
                    <div class="otp-badge">{otp}</div>
                </div>
                <div class="expiry-note">⏱️ This code expires in <b>{OTP_EXPIRY_MINUTES} minutes</b> and can only be used once.</div>
                <p>If you did not request a password reset, you can safely disregard this email. Your existing password will remain secure.</p>
            </div>
            <div class="footer">
                Infosys Springboard Internship 7.0 &bull; Milestone 1: User Authentication Module<br>
                This is an automated system notification.
            </div>
        </div>
    </body>
    </html>
    """

    msg.attach(MIMEText(plain_text, "plain"))
    msg.attach(MIMEText(html_content, "html"))

    try:
        server = smtplib.SMTP("smtp.gmail.com", 587, timeout=15)
        server.starttls()
        server.login(sender_email, app_password)
        server.sendmail(sender_email, clean_to, msg.as_string())
        server.quit()
        return True, "Verification code sent successfully. Please check your email."
    except smtplib.SMTPAuthenticationError:
        return False, "SMTP Authentication failed. Please verify that your Gmail App Password and Email Address are correct."
    except Exception as e:
        return False, f"Failed to send email: {str(e)}"


In [ ]:
%%writefile /content/Milestone1/styles.py
"""
Design system and custom CSS styles for Milestone 1.

Palette:
- Background: Warm White (#FAF9F6)
- Heading Text: Deep Navy (#0F172A)
- Body Text: Slate Navy (#1E293B)
- Muted Text: Steel Slate (#64748B)
- Primary Accent: Deep Teal (#0D9488)
- Hover Accent: Dark Teal (#0F766E)
- Soft Accent: Mint/Teal Tint (#F0FDFA / #CCFBF1)
- Card & Container: Pure White (#FFFFFF) with #E2E8F0 border
"""

import streamlit as st

THEME_COLORS = {
    "bg_main": "#FAF9F6",
    "bg_card": "#FFFFFF",
    "bg_soft": "#F8FAFC",
    "bg_teal_light": "#F0FDFA",
    "text_heading": "#0F172A",
    "text_main": "#1E293B",
    "text_muted": "#64748B",
    "accent_primary": "#0D9488",
    "accent_hover": "#0F766E",
    "accent_border": "#CCFBF1",
    "border_light": "#E2E8F0",
    "border_focus": "#0D9488",
    "success": "#10B981",
    "danger": "#EF4444",
    "warning": "#F59E0B"
}


def inject_custom_styles():
    """Inject cohesive light theme CSS styling into the Streamlit app."""
    css = f"""
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&family=Inter:wght@300;400;500;600;700&display=swap');

        /* Global Reset & Base Typography */
        html, body, [data-testid="stAppViewContainer"] {{
            background-color: {THEME_COLORS['bg_main']} !important;
            font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif !important;
            color: {THEME_COLORS['text_main']} !important;
        }}

        [data-testid="stHeader"] {{
            background: transparent !important;
        }}

        footer {{
            visibility: hidden !important;
            display: none !important;
        }}

        /* Headings */
        h1, h2, h3, h4, h5, h6 {{
            font-family: 'Plus Jakarta Sans', sans-serif !important;
            color: {THEME_COLORS['text_heading']} !important;
            font-weight: 700 !important;
            letter-spacing: -0.02em !important;
        }}

        /* Main Container Width & Spacing */
        .block-container {{
            padding-top: 2.5rem !important;
            padding-bottom: 3.5rem !important;
            max-width: 960px !important;
        }}

        /* Card Container Styling */
        .portal-card {{
            background: {THEME_COLORS['bg_card']};
            border: 1px solid {THEME_COLORS['border_light']};
            border-radius: 14px;
            padding: 32px 36px;
            box-shadow: 0 4px 6px -1px rgba(15, 23, 42, 0.05), 0 2px 4px -2px rgba(15, 23, 42, 0.05);
            margin-bottom: 24px;
        }}

        /* Streamlit Native Bordered Containers */
        div[data-testid="stVerticalBlockBorderWrapper"] {{
            background-color: {THEME_COLORS['bg_card']} !important;
            border: 1px solid {THEME_COLORS['border_light']} !important;
            border-radius: 14px !important;
            padding: 28px !important;
            box-shadow: 0 4px 12px rgba(15, 23, 42, 0.04) !important;
        }}

        /* Form Labels */
        label, [data-testid="stWidgetLabel"] p {{
            font-family: 'Plus Jakarta Sans', sans-serif !important;
            font-size: 13.5px !important;
            font-weight: 600 !important;
            color: {THEME_COLORS['text_heading']} !important;
            margin-bottom: 6px !important;
        }}

        /* Inputs */
        div[data-baseweb="input"] {{
            background-color: {THEME_COLORS['bg_card']} !important;
            border: 1.5px solid {THEME_COLORS['border_light']} !important;
            border-radius: 9px !important;
            transition: all 0.2s ease-in-out !important;
        }}

        div[data-baseweb="input"]:focus-within {{
            border-color: {THEME_COLORS['accent_primary']} !important;
            box-shadow: 0 0 0 3px rgba(13, 148, 136, 0.15) !important;
        }}

        input {{
            font-family: 'Inter', sans-serif !important;
            color: {THEME_COLORS['text_main']} !important;
            font-size: 14.5px !important;
        }}

        /* Select box */
        div[data-baseweb="select"] > div {{
            background-color: {THEME_COLORS['bg_card']} !important;
            border: 1.5px solid {THEME_COLORS['border_light']} !important;
            border-radius: 9px !important;
        }}

        /* Buttons */
        div[data-testid="stButton"] button {{
            background-color: {THEME_COLORS['accent_primary']} !important;
            color: #FFFFFF !important;
            border: 1px solid {THEME_COLORS['accent_primary']} !important;
            border-radius: 9px !important;
            font-family: 'Plus Jakarta Sans', sans-serif !important;
            font-weight: 600 !important;
            font-size: 14px !important;
            height: 44px !important;
            padding: 0 20px !important;
            box-shadow: 0 2px 4px rgba(13, 148, 136, 0.2) !important;
            transition: all 0.2s ease !important;
            width: 100% !important;
        }}

        div[data-testid="stButton"] button:hover {{
            background-color: {THEME_COLORS['accent_hover']} !important;
            border-color: {THEME_COLORS['accent_hover']} !important;
            transform: translateY(-1px) !important;
            box-shadow: 0 4px 8px rgba(13, 148, 136, 0.25) !important;
        }}

        /* Secondary / Outline Button Styling (Used for Cancel / Back) */
        div[data-testid="stButton"].secondary-btn button {{
            background-color: #FFFFFF !important;
            color: {THEME_COLORS['text_muted']} !important;
            border: 1.5px solid {THEME_COLORS['border_light']} !important;
            box-shadow: none !important;
        }}
        div[data-testid="stButton"].secondary-btn button:hover {{
            background-color: {THEME_COLORS['bg_soft']} !important;
            color: {THEME_COLORS['text_heading']} !important;
            border-color: #CBD5E1 !important;
        }}

        /* Alerts and Status Messages */
        div[data-testid="stAlert"] {{
            border-radius: 10px !important;
            font-family: 'Inter', sans-serif !important;
            font-size: 14px !important;
            padding: 12px 16px !important;
        }}

        /* Sidebar Styling */
        section[data-testid="stSidebar"] {{
            background-color: #FFFFFF !important;
            border-right: 1px solid {THEME_COLORS['border_light']} !important;
        }}

        /* Data Tables */
        div[data-testid="stDataFrame"] {{
            border: 1px solid {THEME_COLORS['border_light']};
            border-radius: 10px;
            overflow: hidden;
        }}
    </style>
    """
    st.markdown(css, unsafe_allow_html=True)


def render_portal_header(title: str, subtitle: str = "Infosys Springboard Internship 7.0 &bull; Milestone 1"):
    """Render a clean, distinctive portal header with teal icon and navy typography."""
    st.markdown(f"""
    <div style="text-align: center; margin-bottom: 2rem;">
        <div style="display: inline-flex; align-items: center; justify-content: center; width: 56px; height: 56px; background-color: #F0FDFA; border: 1.5px solid #CCFBF1; border-radius: 14px; margin-bottom: 12px;">
            <span style="font-size: 26px; color: #0D9488;">🛡️</span>
        </div>
        <h1 style="font-size: 1.85rem; margin: 0 0 6px 0; color: #0F172A;">Infosys Authentication Portal</h1>
        <p style="color: #64748B; font-size: 14px; margin: 0; font-weight: 500;">{subtitle}</p>
        <div style="display: inline-block; margin-top: 10px; padding: 4px 14px; background: #F8FAFC; border: 1px solid #E2E8F0; border-radius: 20px; font-size: 12px; font-weight: 600; color: #0D9488;">
            {title}
        </div>
    </div>
    """, unsafe_allow_html=True)


def render_user_badge(username: str, email: str, role: str = "User"):
    """Render user profile badge in the dashboard header."""
    role_color = "#0D9488" if role.lower() == "user" else "#6366F1"
    role_bg = "#F0FDFA" if role.lower() == "user" else "#EEF2FF"
    st.markdown(f"""
    <div style="display: flex; align-items: center; justify-content: space-between; background: #FFFFFF; border: 1px solid #E2E8F0; border-radius: 12px; padding: 18px 24px; margin-bottom: 24px; box-shadow: 0 2px 4px rgba(15,23,42,0.03);">
        <div>
            <div style="font-size: 13px; color: #64748B; font-weight: 500;">Logged in as:</div>
            <div style="font-size: 18px; font-weight: 700; color: #0F172A;">{username} <span style="font-size: 14px; font-weight: 400; color: #64748B;">({email})</span></div>
        </div>
        <div style="background: {role_bg}; color: {role_color}; padding: 6px 14px; border-radius: 20px; font-size: 12px; font-weight: 700; border: 1px solid {role_color}33;">
            ● {role.upper()}
        </div>
    </div>
    """, unsafe_allow_html=True)


In [ ]:
%%writefile /content/Milestone1/app.py
"""
Main Streamlit application for Milestone 1: User Authentication Module.

Meets all Milestone 1 requirements:
- Login via Username or Email with generic error masking
- Signup with 6 mandatory fields, unique username rejection, and redirection to Login
- Strict email format (ab@cd.ef) and password complexity validation
- Single unified SQLite database for users and OTP tracking
- JWT-based session handling
- Forgot Password flow offering both Security Question and Email OTP routes
- User Dashboard with welcome banner and logout
- Separate Admin Login with registered user listing (passwords never exposed)
- Distinctive professional light theme (warm white, deep navy, teal)
"""

import time
import streamlit as st
import pandas as pd

try:
    from Milestone1.auth import (
        create_session_jwt,
        hash_password,
        hash_security_answer,
        validate_email,
        validate_mandatory_fields,
        validate_password,
        validate_password_confirmation,
        verify_password,
        verify_security_answer,
        verify_session_jwt,
    )
    from Milestone1.config import (
        OTP_EXPIRY_MINUTES,
        check_required_secrets_status,
        get_admin_credentials,
    )
    from Milestone1.database import (
        SECURITY_QUESTIONS,
        create_user,
        get_all_registered_users,
        get_user_by_email,
        get_user_by_identifier,
        get_user_by_username,
        init_db,
        update_password,
    )
    from Milestone1.otp_service import (
        create_otp_session,
        generate_otp,
        send_otp_email,
        verify_otp_session,
    )
    from Milestone1.styles import (
        inject_custom_styles,
        render_portal_header,
        render_user_badge,
    )
except ImportError:
    from auth import (
        create_session_jwt,
        hash_password,
        hash_security_answer,
        validate_email,
        validate_mandatory_fields,
        validate_password,
        validate_password_confirmation,
        verify_password,
        verify_security_answer,
        verify_session_jwt,
    )
    from config import (
        OTP_EXPIRY_MINUTES,
        check_required_secrets_status,
        get_admin_credentials,
    )
    from database import (
        SECURITY_QUESTIONS,
        create_user,
        get_all_registered_users,
        get_user_by_email,
        get_user_by_identifier,
        get_user_by_username,
        init_db,
        update_password,
    )
    from otp_service import (
        create_otp_session,
        generate_otp,
        send_otp_email,
        verify_otp_session,
    )
    from styles import (
        inject_custom_styles,
        render_portal_header,
        render_user_badge,
    )


# ── 1. Page Configuration & Theme Setup ─────────────────────────────────
st.set_page_config(
    page_title="Infosys Authentication Portal",
    page_icon="🛡️",
    layout="centered",
    initial_sidebar_state="collapsed"
)

# Initialize database tables on startup
init_db()

# Inject distinctive light theme CSS
inject_custom_styles()


# ── 2. Session State Initialization ─────────────────────────────────────
SESSION_DEFAULTS = {
    "auth_token": None,
    "page": "Login",
    "admin_auth": False,
    # Forgot password state
    "forgot_method": "sq",          # "sq" or "otp"
    "forgot_stage": "identify",      # "identify", "answer", "otp_verify", "reset"
    "forgot_identity": "",
    "forgot_email": "",
    "forgot_sq_text": "",
    "forgot_otp_jwt": "",
    "feedback_message": None,
    "feedback_type": "info"
}

for key, val in SESSION_DEFAULTS.items():
    if key not in st.session_state:
        st.session_state[key] = val


def navigate_to(page_name: str):
    """Safely change page and trigger a rerun."""
    st.session_state.page = page_name
    st.rerun()


def reset_forgot_flow():
    """Reset forgot password state variables."""
    st.session_state.forgot_stage = "identify"
    st.session_state.forgot_identity = ""
    st.session_state.forgot_email = ""
    st.session_state.forgot_sq_text = ""
    st.session_state.forgot_otp_jwt = ""


# ── 3. Application Routing ──────────────────────────────────────────────

# A. Admin Dashboard View
if st.session_state.admin_auth and st.session_state.page == "Admin_Dashboard":
    render_portal_header("Admin Control Panel", "Administrator Access & Audit")
    admin_user, _ = get_admin_credentials()
    render_user_badge(username=admin_user, email="admin@system.local", role="Administrator")

    with st.container():
        st.markdown("### 📋 Registered Users Registry")
        st.caption("Milestone Step 11: Displaying all registered users (passwords and hashes are strictly masked).")

        registered_users = get_all_registered_users()
        if registered_users:
            df = pd.DataFrame(registered_users)
            # Display only safe columns
            display_df = df[["id", "username", "email", "created_at"]].rename(
                columns={
                    "id": "ID",
                    "username": "Username",
                    "email": "Registered Email",
                    "created_at": "Registration Date (UTC)"
                }
            )
            st.dataframe(display_df, use_container_width=True, hide_index=True)
            st.markdown(f"**Total Registered Users:** `{len(registered_users)}`")
        else:
            st.info("No registered users found in the database. Use the Signup page to register test accounts.")

        st.markdown("<br>", unsafe_allow_html=True)
        if st.button("🚪 Logout from Admin Dashboard", use_container_width=True):
            st.session_state.admin_auth = False
            st.session_state.page = "Login"
            st.rerun()

# B. Regular User Dashboard View (JWT Session Protected)
elif st.session_state.auth_token and st.session_state.page == "Dashboard":
    is_valid, payload, err = verify_session_jwt(st.session_state.auth_token)
    if not is_valid or not payload:
        st.warning("Your session has expired or is invalid. Please sign in again.")
        st.session_state.auth_token = None
        st.session_state.page = "Login"
        st.rerun()

    username = payload.get("username", "User")
    email = payload.get("email", "unknown")

    render_portal_header("User Dashboard", "Secure Authenticated Session")
    render_user_badge(username=username, email=email, role="User")

    with st.container():
        st.markdown(f"### 👋 Welcome back, **{username}**!")
        st.markdown(
            "You have successfully authenticated via JWT. Your session token is active and valid for 2 hours."
        )

        col1, col2, col3 = st.columns(3)
        with col1:
            st.metric(label="Authentication Status", value="Verified", delta="Active JWT")
        with col2:
            st.metric(label="Security Level", value="Bcrypt + Salt", delta="HS256")
        with col3:
            st.metric(label="Session Expiry", value="2 Hours", delta="Standard")

        st.markdown("<br>", unsafe_allow_html=True)
        st.markdown("#### 🔒 Account Details")
        st.write(f"- **Username:** `{username}`")
        st.write(f"- **Primary Email:** `{email}`")
        st.write(f"- **Session Token Type:** `Bearer JWT (HS256)`")

        st.markdown("<br><hr style='border-color: #E2E8F0;'><br>", unsafe_allow_html=True)
        if st.button("🚪 Logout", use_container_width=True):
            st.session_state.auth_token = None
            st.session_state.page = "Login"
            st.success("You have been securely logged out.")
            time.sleep(1)
            st.rerun()

# C. Admin Login View (Step 11)
elif st.session_state.page == "Admin_Login":
    render_portal_header("Admin Portal Login", "Restricted Access for Administrators")

    with st.container():
        st.markdown("Enter dedicated administrator credentials to access the user registry:")
        admin_u_input = st.text_input("Admin Username", placeholder="e.g. admin")
        admin_p_input = st.text_input("Admin Password", type="password", placeholder="••••••••")

        st.markdown("<br>", unsafe_allow_html=True)
        col_btn1, col_btn2 = st.columns(2)
        with col_btn1:
            if st.button("Sign In as Admin →", use_container_width=True):
                cfg_user, cfg_pass = get_admin_credentials()
                if admin_u_input.strip() == cfg_user and admin_p_input == cfg_pass:
                    st.session_state.admin_auth = True
                    st.session_state.page = "Admin_Dashboard"
                    st.success("Admin authentication successful.")
                    st.rerun()
                else:
                    st.error("❌ Invalid admin credentials.")
        with col_btn2:
            if st.button("← Return to User Login", use_container_width=True):
                st.session_state.page = "Login"
                st.rerun()

# D. Signup View (Step 3, 7, 8)
elif st.session_state.page == "Signup":
    render_portal_header("Create New Account", "Register for Infosys Authentication Portal")

    with st.container():
        st.markdown("Please fill in all mandatory fields to create your account:")

        signup_username = st.text_input("Username *", placeholder="e.g. alex_kumar")
        signup_email = st.text_input("Email Address *", placeholder="e.g. alex@infosys.com")
        signup_password = st.text_input(
            "Password *",
            type="password",
            placeholder="Min 8 chars, 1 uppercase, 1 lowercase, 1 number, 1 symbol"
        )
        signup_confirm_password = st.text_input(
            "Confirm Password *",
            type="password",
            placeholder="Re-enter your password"
        )
        signup_sq = st.selectbox(
            "Security Question * (Used for password recovery)",
            SECURITY_QUESTIONS
        )
        signup_sa = st.text_input(
            "Security Answer *",
            placeholder="Your confidential answer"
        )

        st.markdown("<br>", unsafe_allow_html=True)
        col_sub1, col_sub2 = st.columns(2)

        with col_sub1:
            if st.button("Complete Registration →", use_container_width=True):
                # 1. Check mandatory fields
                ok_req, msg_req = validate_mandatory_fields(
                    username=signup_username,
                    email=signup_email,
                    password=signup_password,
                    confirm_password=signup_confirm_password,
                    security_answer=signup_sa
                )
                if not ok_req:
                    st.error(f"⚠️ {msg_req}")
                else:
                    # 2. Validate email format
                    ok_email, msg_email = validate_email(signup_email)
                    if not ok_email:
                        st.error(f"⚠️ {msg_email}")
                    else:
                        # 3. Validate password complexity
                        ok_pass, msg_pass = validate_password(signup_password)
                        if not ok_pass:
                            st.error(f"⚠️ {msg_pass}")
                        else:
                            # 4. Validate password confirmation
                            ok_match, msg_match = validate_password_confirmation(signup_password, signup_confirm_password)
                            if not ok_match:
                                st.error(f"❌ {msg_match}")
                            else:
                                # 5. Hash credentials securely
                                pwd_hash = hash_password(signup_password)
                                sa_hash = hash_security_answer(signup_sa)

                                # 6. Persist to SQLite
                                success, db_msg = create_user(
                                    username=signup_username,
                                    email=signup_email,
                                    password_hash=pwd_hash,
                                    security_question=signup_sq,
                                    security_answer_hash=sa_hash
                                )
                                if success:
                                    # Milestone Step 7 rule: Route back to Login on success!
                                    st.success("✅ Account created successfully! Please sign in with your credentials.")
                                    time.sleep(1.5)
                                    navigate_to("Login")
                                else:
                                    st.error(f"❌ {db_msg}")

        with col_sub2:
            if st.button("← Already have an account? Sign In", use_container_width=True):
                navigate_to("Login")

# E. Forgot Password View (Step 8 & 9)
elif st.session_state.page == "Forgot":
    render_portal_header("Reset Password", "Two Secure Identity Verification Routes")

    with st.container():
        # Recovery Route Switcher Tabs
        route_tab1, route_tab2 = st.tabs(["🔒 Route 1: Security Question", "📧 Route 2: Email OTP"])

        # ── ROUTE 1: SECURITY QUESTION ──
        with route_tab1:
            st.markdown("#### Recover via Security Question")
            st.caption("Enter your username to answer the security question you selected at signup.")

            sq_uname = st.text_input("Registered Username", key="sq_uname_input", placeholder="e.g. alex_kumar")

            if st.button("Find Security Question →", key="sq_find_btn"):
                if not sq_uname.strip():
                    st.error("⚠️ Please enter your username.")
                else:
                    user = get_user_by_username(sq_uname.strip())
                    if not user:
                        st.error("❌ Username not found in the registry.")
                    else:
                        st.session_state.forgot_identity = user["username"]
                        st.session_state.forgot_sq_text = user["security_question"]
                        st.session_state.forgot_stage = "answer"

            if st.session_state.forgot_stage == "answer" and st.session_state.forgot_identity:
                st.info(f"❓ **Security Question:** {st.session_state.forgot_sq_text}")
                sq_ans_input = st.text_input("Your Security Answer", type="password", key="sq_ans_input")
                sq_new_pwd = st.text_input("New Password", type="password", key="sq_new_pwd_input",
                                           placeholder="Min 8 chars, uppercase, lowercase, digit, symbol")
                sq_confirm_pwd = st.text_input("Confirm New Password", type="password", key="sq_conf_pwd_input")

                if st.button("Verify Answer & Reset Password →", key="sq_submit_btn"):
                    ok_flds, msg_flds = validate_mandatory_fields(
                        security_answer=sq_ans_input,
                        new_password=sq_new_pwd,
                        confirm_password=sq_confirm_pwd
                    )
                    if not ok_flds:
                        st.error(f"⚠️ {msg_flds}")
                    else:
                        user = get_user_by_username(st.session_state.forgot_identity)
                        if not user or not verify_security_answer(sq_ans_input, user["security_answer_hash"]):
                            st.error("❌ Incorrect security answer.")
                        else:
                            ok_pwd, msg_pwd = validate_password(sq_new_pwd)
                            if not ok_pwd:
                                st.error(f"⚠️ {msg_pwd}")
                            else:
                                ok_m, msg_m = validate_password_confirmation(sq_new_pwd, sq_confirm_pwd)
                                if not ok_m:
                                    st.error(f"❌ {msg_m}")
                                else:
                                    new_hash = hash_password(sq_new_pwd)
                                    update_password(st.session_state.forgot_identity, new_hash)
                                    st.success("✅ Password reset successfully! Redirecting to login...")
                                    reset_forgot_flow()
                                    time.sleep(1.5)
                                    navigate_to("Login")

        # ── ROUTE 2: EMAIL OTP ──
        with route_tab2:
            st.markdown("#### Recover via One-Time Password (OTP)")
            st.caption(f"Receive a cryptographically secure 6-digit code via Gmail (expires in {OTP_EXPIRY_MINUTES} mins).")

            otp_email_input = st.text_input("Registered Email Address", key="otp_email_input", placeholder="e.g. alex@infosys.com")

            if st.button("Send Verification Code 📧", key="otp_send_btn"):
                ok_req, msg_req = validate_mandatory_fields(email=otp_email_input)
                if not ok_req:
                    st.error(f"⚠️ {msg_req}")
                else:
                    ok_fmt, msg_fmt = validate_email(otp_email_input)
                    if not ok_fmt:
                        st.error(f"⚠️ {msg_fmt}")
                    else:
                        user = get_user_by_email(otp_email_input.strip().lower())
                        if not user:
                            st.error("❌ Email address is not registered in our system.")
                        else:
                            # Generate OTP and 5-minute session
                            otp_code = generate_otp()
                            jwt_otp_token, _ = create_otp_session(user["email"], otp_code)

                            with st.spinner("Dispatching verification email via Gmail SMTP..."):
                                email_sent, email_msg = send_otp_email(user["email"], otp_code)

                            if email_sent:
                                st.session_state.forgot_email = user["email"]
                                st.session_state.forgot_otp_jwt = jwt_otp_token
                                st.session_state.forgot_stage = "otp_verify"
                                st.success("✅ 6-digit code sent to your email! Please enter it below.")
                            else:
                                st.error(f"❌ {email_msg}")

            if st.session_state.forgot_stage == "otp_verify" and st.session_state.forgot_email:
                st.info(f"📩 Verification code sent to **{st.session_state.forgot_email}**. Enter it within {OTP_EXPIRY_MINUTES} minutes.")
                entered_otp = st.text_input("Enter 6-Digit Code", max_chars=6, key="entered_otp_input", placeholder="e.g. 583921")
                otp_new_pwd = st.text_input("New Password", type="password", key="otp_new_pwd_input",
                                            placeholder="Min 8 chars, uppercase, lowercase, digit, symbol")
                otp_confirm_pwd = st.text_input("Confirm New Password", type="password", key="otp_conf_pwd_input")

                if st.button("Verify Code & Reset Password →", key="otp_verify_btn"):
                    ok_flds, msg_flds = validate_mandatory_fields(
                        verification_code=entered_otp,
                        new_password=otp_new_pwd,
                        confirm_password=otp_confirm_pwd
                    )
                    if not ok_flds:
                        st.error(f"⚠️ {msg_flds}")
                    else:
                        # Verify OTP against single-use DB record and 5-min JWT
                        is_otp_ok, otp_status_msg = verify_otp_session(
                            jwt_token=st.session_state.forgot_otp_jwt,
                            input_otp=entered_otp,
                            target_email=st.session_state.forgot_email
                        )
                        if not is_otp_ok:
                            st.error(f"❌ {otp_status_msg}")
                        else:
                            ok_pwd, msg_pwd = validate_password(otp_new_pwd)
                            if not ok_pwd:
                                st.error(f"⚠️ {msg_pwd}")
                            else:
                                ok_m, msg_m = validate_password_confirmation(otp_new_pwd, otp_confirm_pwd)
                                if not ok_m:
                                    st.error(f"❌ {msg_m}")
                                else:
                                    new_hash = hash_password(otp_new_pwd)
                                    update_password(st.session_state.forgot_email, new_hash)
                                    st.success("✅ Password successfully updated! Redirecting to login...")
                                    reset_forgot_flow()
                                    time.sleep(1.5)
                                    navigate_to("Login")

        st.markdown("<br><hr style='border-color: #E2E8F0;'><br>", unsafe_allow_html=True)
        if st.button("← Cancel and Return to Sign In", use_container_width=True):
            reset_forgot_flow()
            navigate_to("Login")

# F. Default: Login View (Step 3 & 7)
else:
    render_portal_header("Sign In to Your Account", "Access User Analytics & Dashboard")

    with st.container():
        login_identifier = st.text_input(
            "Username or Email Address *",
            placeholder="e.g. alex_kumar or alex@infosys.com"
        )
        login_password = st.text_input(
            "Password *",
            type="password",
            placeholder="••••••••"
        )

        st.markdown("<br>", unsafe_allow_html=True)
        col_btn_login, col_btn_signup = st.columns([1.2, 1])

        with col_btn_login:
            if st.button("Sign In →", use_container_width=True):
                ok_req, msg_req = validate_mandatory_fields(
                    username_or_email=login_identifier,
                    password=login_password
                )
                if not ok_req:
                    st.error(f"⚠️ {msg_req}")
                else:
                    # Look up user by username OR email
                    user = get_user_by_identifier(login_identifier)

                    # Verify bcrypt hash
                    if user and verify_password(login_password, user["password_hash"]):
                        # Issue fresh JWT session token on login success (Step 7)
                        token = create_session_jwt(
                            username=user["username"],
                            email=user["email"],
                            role="user"
                        )
                        st.session_state.auth_token = token
                        st.session_state.page = "Dashboard"
                        st.success("Authentication successful.")
                        st.rerun()
                    else:
                        # Milestone Step 3: Generic error — never reveal if username or password was wrong!
                        st.error("❌ Invalid username/email or password.")

        with col_btn_signup:
            if st.button("Create Account", use_container_width=True):
                navigate_to("Signup")

        st.markdown("<br>", unsafe_allow_html=True)
        col_forgot, col_admin = st.columns(2)
        with col_forgot:
            if st.button("🔑 Forgot Password?", use_container_width=True):
                reset_forgot_flow()
                navigate_to("Forgot")
        with col_admin:
            if st.button("🛡️ Admin Portal", use_container_width=True):
                navigate_to("Admin_Login")


In [ ]:
# 3. Initialize Database and Run Automated Verification Tests
from database import init_db
init_db()
print("✅ SQLite database initialized.")

# Verify dependencies & core modules
import auth, otp_service, styles
print("✅ All modules imported cleanly.")


In [ ]:
# 4. Launch Streamlit Application via Secure ngrok Tunnel
import subprocess, time
from pyngrok import ngrok
from google.colab import userdata

# Retrieve ngrok token from Colab Secrets
ngrok_token = userdata.get('NGROK_AUTHTOKEN')
if not ngrok_token:
    raise ValueError("NGROK_AUTHTOKEN not found in Colab Secrets! Please configure it in the Secrets tab.")

ngrok.set_auth_token(ngrok_token)

# Terminate any previously running instances
!pkill -f streamlit
ngrok.kill()
time.sleep(2)

# Start Streamlit server in background on port 8501
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)

# Connect ngrok HTTP tunnel
try:
    public_url = ngrok.connect(8501).public_url
    print("=" * 68)
    print(f"🚀 Infosys Authentication Portal Live URL: {public_url}")
    print("=" * 68)
    print("Application is online! Click the link above to test your portal.")
    print("Press the Stop button [■] or Ctrl+C to shut down gracefully.\n")
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nShutting down server and closing tunnels...")
    ngrok.kill()
    process.terminate()
    !pkill -f streamlit
    print("✅ Streamlit and ngrok tunnel stopped successfully.")
